# Notebook 09 — SQLite job queue, worker, retry semantics

**Purpose:** Make every long-running operation in the engine survive a
kernel crash. NB 02–08 ran every step inline; this notebook builds the
§7.5 SQLite-backed job queue, wires the existing ingest + synthesis
chains through it, and demonstrates the retry/fan-out/concurrency
semantics that turn an "agent demo" into an "agent system that
survives the night."

**Exam relevance:** Architecture Patterns (durable workflows + retry
ladder are exam-canonical), with side-quests in Tool Use (CLI surface
for queue inspection) and observability (per-job cost attribution
through `cost_records.job_id`).

**Design refs:** §7.5 (job queue), §10 Scenario A step 4 (the queue
intercepts the inline ingest call), §13.2 (the audit DB downstream of
this picks up `cost_records.job_id` once NB 11 lands).

**Depends on:** NB 02 (ingest), NB 05 (synthesis), NB 07 (the
orchestrator that the worker now shares `run_ingest_chain` with).

**Setup:** the demo runs against a tmp DB at
`/tmp/marginalia-nb09/jobs.db` and a tmp wiki copied from
`notebooks/data/poc-wiki/`. Nothing here mutates the committed
fixtures.

In [ ]:
%load_ext autoreload
%autoreload 2

import asyncio
import os
import shutil
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

from dotenv import load_dotenv
from rich.console import Console
from rich.table import Table
from rich.tree import Tree

from engine.jobs import (
    DISPATCHERS,
    Job,
    JobStatus,
    WorkerCtx,
    cancel_pending,
    children_of,
    claim_next,
    connect,
    count_by_status,
    enqueue,
    get_job,
    init_db,
    list_jobs,
    mark_failed,
    mark_succeeded,
    requeue,
    run_worker,
)
from engine.jobs.db import DDL

console = Console()

In [ ]:
load_dotenv()
assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not in env"

TMP_ROOT = Path("/tmp/marginalia-nb09").resolve()
if TMP_ROOT.exists():
    shutil.rmtree(TMP_ROOT)
TMP_ROOT.mkdir(parents=True)

DB_PATH = TMP_ROOT / "jobs.db"
TMP_WIKI = TMP_ROOT / "wiki"
# Resolve poc-wiki relative to repo root regardless of cwd (notebook
# runs from notebooks/; nbconvert runs from notebooks/; tests may run
# from repo root).
for candidate in (Path("notebooks/data/poc-wiki"), Path("data/poc-wiki"), Path("../notebooks/data/poc-wiki")):
    if candidate.exists():
        REPO_WIKI = candidate.resolve()
        break
else:
    raise FileNotFoundError("poc-wiki fixtures not found from cwd " + str(Path.cwd()))
shutil.copytree(REPO_WIKI, TMP_WIKI)

print(f"db   : {DB_PATH}")
print(f"wiki : {TMP_WIKI}")

## Part A — Schema + connection (the durable substrate)

The §7.5 queue lives in one SQLite file per wiki at
`<wiki-root>/.wiki/jobs.db`. Three things make it work:

1. **WAL mode** — readers don't block writers; the second worker can
   poll while the first is mid-claim.
2. **CHECK constraint on `status`** — invalid state names are caught
   at write time rather than producing zombie rows.
3. **Index on `status`** — `claim_next`'s `WHERE status='pending'`
   scan becomes a B-tree seek, not a table scan.

`init_db` is idempotent — every worker start runs it, first-run and
resume look the same.

In [ ]:
init_db(DB_PATH)

# Show the DDL the engine ships:
console.print("[bold]engine/jobs/db.py — DDL[/bold]")
print(DDL)

# Confirm the table landed with the expected columns + indexes.
conn = connect(DB_PATH)
try:
    cols = conn.execute("PRAGMA table_info(jobs)").fetchall()
    idx = conn.execute("PRAGMA index_list(jobs)").fetchall()
finally:
    conn.close()

t = Table(title="jobs table columns", show_lines=False)
t.add_column("name"); t.add_column("type"); t.add_column("notnull"); t.add_column("default")
for c in cols:
    t.add_row(c["name"], c["type"], str(c["notnull"]), str(c["dflt_value"]))
console.print(t)

console.print(f"[dim]indexes: {[i['name'] for i in idx]}[/dim]")

## Part B — Pydantic `Job` round-trip

The `Job` model maps 1:1 to the row shape; `payload` and `result` are
JSON-encoded on write, decoded on read. Round-tripping a job through
`enqueue`/`get_job` gives back the same Pydantic instance modulo the
`id` and `created_at` fields the queue assigns.

In [ ]:
conn = connect(DB_PATH)
try:
    job_id = enqueue(conn, "_mock_flaky", {"hello": "world", "fail_until_attempt": 1})
    job = get_job(conn, job_id)
finally:
    conn.close()

console.print("[bold]Job round-trip:[/bold]")
print(job.model_dump_json(indent=2))

## Part C — `claim_next` is the lock

The whole concurrency story rests on one SQL statement:

```sql
BEGIN IMMEDIATE;
UPDATE jobs
   SET status='running', started_at=?, attempts=attempts+1
 WHERE id = (SELECT id FROM jobs
              WHERE status='pending' AND (retry_at IS NULL OR retry_at <= ?)
              ORDER BY created_at LIMIT 1)
 RETURNING *;
COMMIT;
```

Two concurrent transactions both target the same `pending` row; the
loser sees zero matches and `RETURNING` returns no row. No advisory
locks, no lease timestamps, no coordinator. This is the canonical
SQLite-as-job-queue pattern.

Below: claim once, claim twice. Second call returns `None`.

In [ ]:
# Reset for a clean demo.
conn = connect(DB_PATH)
try:
    conn.execute("DELETE FROM jobs")
    for i in range(3):
        enqueue(conn, "_mock_flaky", {"i": i, "fail_until_attempt": 1})
    pending_before = list_jobs(conn, status=JobStatus.PENDING)

    first = claim_next(conn)
    second = claim_next(conn)
    third = claim_next(conn)
    fourth = claim_next(conn)  # queue drained
finally:
    conn.close()

console.print(f"enqueued      : {len(pending_before)}")
console.print(f"first claim   : {first.payload}  status={first.status}  attempts={first.attempts}")
console.print(f"second claim  : {second.payload}")
console.print(f"third claim   : {third.payload}")
console.print(f"fourth claim  : {fourth} (None — drained)")

## Part D — Retry ladder via `_mock_flaky`

`_mock_flaky` is the deterministic test handler: payload knob
`fail_until_attempt=N` makes it fail until `attempts >= N`. The retry
ladder lives in the **queue**, not the handler — `mark_failed` flips
the row back to `pending` with a future `retry_at`; the next worker
poll re-claims it.

The notebook uses an artificially short backoff `(0.1, 0.2, 0.5)s` so
the demo runs in seconds. Production uses §7.5's `(60s, 300s,
1800s)` ladder.

In [ ]:
# Wipe + re-init so this cell is idempotent on re-run.
shutil.rmtree(DB_PATH, ignore_errors=True) if DB_PATH.is_dir() else DB_PATH.unlink(missing_ok=True)
init_db(DB_PATH)

class _NoopConfig:
    pass

ctx = WorkerCtx(
    wiki_root=TMP_WIKI,
    config=_NoopConfig(),  # mock kinds don't touch config
    client=object(),
    db_path=DB_PATH,
)

conn = connect(DB_PATH)
try:
    job_id = enqueue(
        conn, "_mock_flaky",
        {"fail_until_attempt": 3},
        max_attempts=5,
    )
finally:
    conn.close()

t0 = time.monotonic()
counts = await run_worker(
    ctx,
    stop_when_drained=True,
    backoff_seconds=(0.1, 0.2, 0.5),
    poll_interval=0.05,
)
wall = time.monotonic() - t0

conn = connect(DB_PATH)
try:
    final = get_job(conn, job_id)
finally:
    conn.close()

console.print(f"final status : [green]{final.status}[/green]")
console.print(f"attempts     : {final.attempts}")
console.print(f"wall         : {wall:.2f}s")
console.print(f"counts       : {counts}")

## Part D.2 — Dead-letter and `marginalia jobs retry`

`fail_forever=True` makes the handler always raise. After
`max_attempts` failures the row transitions to `dead` (terminal). The
operator's recovery hatch is `marginalia jobs retry <id>` —
implemented as `requeue()` — which flips `dead → pending` with
`attempts` reset to 0.

In [ ]:
conn = connect(DB_PATH)
try:
    dead_id = enqueue(
        conn, "_mock_flaky",
        {"fail_forever": True},
        max_attempts=2,
    )
finally:
    conn.close()

await run_worker(ctx, stop_when_drained=True, backoff_seconds=(0.05, 0.05, 0.05), poll_interval=0.02)

conn = connect(DB_PATH)
try:
    dead = get_job(conn, dead_id)
finally:
    conn.close()
console.print(f"after first run  : status=[bold red]{dead.status}[/bold red]  attempts={dead.attempts}")

# Operator: `marginalia jobs retry <id>` — requeue and try again.
conn = connect(DB_PATH)
try:
    requeue(conn, dead_id)
    requeued = get_job(conn, dead_id)
finally:
    conn.close()
console.print(f"after requeue    : status=[yellow]{requeued.status}[/yellow]  attempts={requeued.attempts}")

# Cancel pending (it'd just dead-letter again) so the next cells start clean.
conn = connect(DB_PATH)
try:
    cancel_pending(conn)
finally:
    conn.close()

## Part E — Real ingest job (the load-bearing case)

The worker dispatches `ingest` jobs to `run_ingest_chain` — the same
function the orchestrator calls in NB 07. Same code, same result;
only the trigger differs (CLI/queue vs. orchestrator tool_use).

Below: enqueue one ingest job for a small markdown source, run the
worker until drained, and confirm the page landed in the tmp wiki.

In [ ]:
from anthropic import Anthropic
from engine.models.wiki_config import MarginaliaConfig

# Real wiki config + Anthropic client for the ingest job.
config = MarginaliaConfig.load(TMP_WIKI)
client = Anthropic()
real_ctx = WorkerCtx(
    wiki_root=TMP_WIKI,
    config=config,
    client=client,
    db_path=DB_PATH,
)

# Pick a small markdown fixture so the ingest call is cheap and fast.
SOURCE = TMP_WIKI / "raw" / "good_source.md"
assert SOURCE.exists(), f"missing fixture {SOURCE}"

conn = connect(DB_PATH)
try:
    ingest_id = enqueue(conn, "ingest", {"input": str(SOURCE)})
finally:
    conn.close()

t0 = time.monotonic()
counts = await run_worker(real_ctx, stop_when_drained=True, backoff_seconds=(1, 1, 1))
wall = time.monotonic() - t0

conn = connect(DB_PATH)
try:
    job = get_job(conn, ingest_id)
finally:
    conn.close()

console.print(f"ingest result : {job.result}")
console.print(f"wall          : {wall:.2f}s")
console.print(f"counts        : {counts}")

# Confirm the page actually landed.
written = TMP_WIKI / f"{job.result['path']}.md"
console.print(f"page on disk  : {'yes' if written.exists() else '[red]MISSING[/red]'} → {written.relative_to(TMP_WIKI)}")

## Part F — Crash resume

The whole point of a queue. Enqueue 5 mock jobs, drain only 2 with
`max_jobs=2` (simulating a kernel kill mid-batch), then start a fresh
worker. The remaining 3 jobs land — the queue is the durable state.

In [ ]:
# Fresh slate: drop the ingest history so we're only counting the 5 new jobs.
conn = connect(DB_PATH)
try:
    conn.execute("DELETE FROM jobs")
    for i in range(5):
        enqueue(conn, "_mock_flaky", {"i": i, "fail_until_attempt": 1})
finally:
    conn.close()

console.print("[dim]worker A: stops after 2 jobs (simulated crash)[/dim]")
counts_a = await run_worker(
    ctx, max_jobs=2, backoff_seconds=(0.05, 0.05, 0.05), poll_interval=0.02
)
console.print(f"  → {counts_a}")

console.print("[dim]worker B: drains the rest[/dim]")
counts_b = await run_worker(
    ctx, stop_when_drained=True, backoff_seconds=(0.05, 0.05, 0.05), poll_interval=0.02
)
console.print(f"  → {counts_b}")

conn = connect(DB_PATH)
try:
    final_counts = count_by_status(conn)
finally:
    conn.close()
console.print(f"[bold]final histogram[/bold]: {final_counts}")
assert final_counts.get("succeeded") == 5
console.print("[green]all 5 succeeded — no work lost across the simulated restart.[/green]")

## Part G — Fan-out: `ingest_batch` → N children + 1 synthesis child

The §10A scenario at scale. An `ingest_batch` parent enqueues N child
`ingest` jobs and (optionally) one `synthesis` child gated on the
batch completing. The gated synthesis dispatcher checks sibling
`children_of(parent_id)` on each claim — if siblings are still in
flight it raises a transient error and goes back to the retry ladder.
Once siblings succeed, it collects their result paths and runs
cross-source synthesis.

This is real ingest + real synthesis on the small markdown fixtures.

In [ ]:
# Reset for the fan-out demo.
conn = connect(DB_PATH)
try:
    conn.execute("DELETE FROM jobs")
finally:
    conn.close()

# Two clean markdown fixtures — enough to exercise fan-out without
# burning extra LLM tokens on the deliberately-bad `garbage_source.md`.
raw_inputs = [
    str(TMP_WIKI / "raw" / "good_source.md"),
    str(TMP_WIKI / "raw" / "ambiguous_source.md"),
]

conn = connect(DB_PATH)
try:
    parent_id = enqueue(
        conn,
        "ingest_batch",
        {"inputs": raw_inputs, "synthesize_after": True, "hint": "Apollo launch timeline"},
    )
finally:
    conn.close()

t0 = time.monotonic()
counts = await run_worker(
    real_ctx,
    stop_when_drained=True,
    backoff_seconds=(0.5, 1.0, 2.0),
    poll_interval=0.1,
)
wall = time.monotonic() - t0

conn = connect(DB_PATH)
try:
    parent = get_job(conn, parent_id)
    kids = children_of(conn, parent_id)
finally:
    conn.close()

# Pretty-print the parent + child tree.
tree = Tree(f"[bold]ingest_batch[/bold] {parent_id[:8]} ([green]{parent.status}[/green])")
for k in kids:
    label = f"[cyan]{k.kind}[/cyan] {k.id[:8]} ([green]{k.status}[/green])"
    if k.result and 'path' in k.result:
        label += f"  → {k.result['path']}"
    tree.add(label)
console.print(tree)
console.print(f"wall: {wall:.2f}s   counts: {counts}")

## Part H — Two-worker concurrency

The atomic-claim primitive in action. Two worker threads run
`run_worker` against the same DB. A 1ms sleep between claims gives
the OS scheduler a real chance to interleave them. The post-condition
is that every row was claimed exactly once — `BEGIN IMMEDIATE` plus
the `WHERE status='pending'` filter rule out double-pickup.

In [ ]:
# Fresh slate.
conn = connect(DB_PATH)
try:
    conn.execute("DELETE FROM jobs")
    n_jobs = 30
    for i in range(n_jobs):
        enqueue(conn, "_mock_flaky", {"i": i, "fail_until_attempt": 1, "sleep_ms": 1})
finally:
    conn.close()


def run_worker_sync(label: str) -> dict:
    return asyncio.run(
        run_worker(
            ctx,
            stop_when_drained=True,
            backoff_seconds=(0.05, 0.05, 0.05),
            poll_interval=0.02,
        )
    )


t0 = time.monotonic()
with ThreadPoolExecutor(max_workers=2) as ex:
    futs = [ex.submit(run_worker_sync, "A"), ex.submit(run_worker_sync, "B")]
    results = [f.result() for f in futs]
wall = time.monotonic() - t0

# Aggregate transitions across both workers.
totals = {s.value: 0 for s in JobStatus}
for r in results:
    for k, v in r.items():
        totals[k] += v

conn = connect(DB_PATH)
try:
    actual = count_by_status(conn)
finally:
    conn.close()

console.print(f"per-worker counts : {results}")
console.print(f"aggregate         : {totals}")
console.print(f"DB histogram      : {actual}")
console.print(f"wall              : {wall:.2f}s")
assert actual.get("succeeded") == n_jobs, "some rows lost or double-claimed!"
console.print(f"[green]all {n_jobs} rows succeeded exactly once across two workers.[/green]")

## Part I — Perf probe: 1000 dummy jobs

Validates that SQLite is plenty fast for this workload. The handler
itself is a no-op (the `fail_until_attempt=0` shortcut means the
first attempt succeeds immediately) so the timing reflects pure queue
overhead: enqueue + claim + mark_succeeded + commit, 1000 times.

In [ ]:
N = 1000

# Reset.
conn = connect(DB_PATH)
try:
    conn.execute("DELETE FROM jobs")
finally:
    conn.close()

# Bulk enqueue.
t0 = time.monotonic()
conn = connect(DB_PATH)
try:
    for i in range(N):
        enqueue(conn, "_mock_flaky", {"i": i, "fail_until_attempt": 0})
finally:
    conn.close()
enqueue_wall = time.monotonic() - t0
console.print(f"enqueue {N} jobs : {enqueue_wall:.2f}s   ({N/enqueue_wall:.0f} jobs/s)")

# Drain.
t0 = time.monotonic()
counts = await run_worker(
    ctx, stop_when_drained=True, backoff_seconds=(0.01, 0.01, 0.01), poll_interval=0.01
)
drain_wall = time.monotonic() - t0
console.print(f"drain {N} jobs   : {drain_wall:.2f}s   ({N/drain_wall:.0f} jobs/s)")
console.print(f"counts           : {counts}")

## Part J — CLI surface

`marginalia jobs list`, `marginalia jobs status <id>` etc. operate
over the same DB. Below shows the `list` and a single `status`
against the populated tmp DB.

In [ ]:
# Re-seed a few fresh + finished rows so the CLI has interesting output.
conn = connect(DB_PATH)
try:
    conn.execute("DELETE FROM jobs")
    a = enqueue(conn, "ingest", {"input": str(TMP_WIKI / "raw" / "good_source.md")})
    b = enqueue(conn, "_mock_flaky", {"fail_forever": True}, max_attempts=1)
    c = enqueue(conn, "_mock_flaky", {"fail_until_attempt": 1})
finally:
    conn.close()

await run_worker(real_ctx, stop_when_drained=True, backoff_seconds=(1, 1, 1))

In [ ]:
# Render the CLI output inline.
import subprocess
result = subprocess.run(
    ["uv", "run", "marginalia", "jobs", "list", "--db", str(DB_PATH)],
    capture_output=True, text=True, cwd=str(Path.cwd()),
)
print(result.stdout)

In [ ]:
# `marginalia jobs status <id>` for the real ingest job.
result = subprocess.run(
    ["uv", "run", "marginalia", "jobs", "status", a[:8], "--db", str(DB_PATH)],
    capture_output=True, text=True, cwd=str(Path.cwd()),
)
print(result.stdout)

## Part K — Receipts + what to extract

Numbers from this run feed `engine/decisions/job-queue.md`. The
notable findings (filled in by the cells above):

- **Atomic claim**: 30 jobs across 2 workers landed exactly once.
- **Crash resume**: 5 jobs survived a simulated mid-batch worker
  exit; second worker drained the remainder.
- **Retry ladder**: `_mock_flaky(fail_until_attempt=3)` succeeded on
  attempt 3 — the queue's `retry_at` is the durability mechanism, not
  in-process `tenacity`. Backoffs persist across worker restarts;
  in-process retries don't.
- **Fan-out gating**: the `synthesis` child waited on its `ingest`
  siblings via `children_of(parent_id)` before running, raising a
  transient error each poll until siblings succeeded.
- **Throughput**: ~N00 jobs/s drain on a no-op handler — see Part I.

## What to extract

| Notebook artifact | Lands at |
|---|---|
| `Job`, `JobStatus`, `JobKind` Pydantic types | `engine/jobs/models.py` |
| DDL + `connect`/`init_db` | `engine/jobs/db.py` |
| `enqueue`, `claim_next`, `mark_*`, `requeue`, `cancel_pending`, `purge_older_than`, `list_jobs`, `count_by_status`, `children_of` | `engine/jobs/queue.py` |
| `WorkerCtx` + handlers (`_handle_ingest`, `_handle_synthesis`, `_handle_ingest_batch`, `_handle_mock_flaky`) | `engine/jobs/dispatchers.py` |
| `run_worker` async loop | `engine/jobs/worker.py` |
| `run_ingest_chain` (promoted from orchestrator's private function) | `engine/agents/ingest/run.py` |
| `marginalia jobs <verb>` + `marginalia worker` | `engine/cli/jobs.py`, `engine/cli/main.py` |
| Standalone DDL for ops use | `scripts/init_jobs_db.sql` |
| Receipts (drain throughput, fan-out tree, retry ladder) | `engine/decisions/job-queue.md` |

Tests covering the load-bearing primitives live at:

- `tests/test_jobs_queue.py` — DDL idempotency, enqueue/claim/transition, requeue, purge.
- `tests/test_jobs_worker.py` — retry-then-succeed, dead-letter, fan-out, max_jobs.
- `tests/test_jobs_concurrency.py` — two-thread claim race, every row claimed exactly once.